# 130 — Agent Skills como capacidades portables

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Metadatos = 60 × 35 = 2 100; activados = 400 + 700 + 1 200 = 2 300;
total = **4 400 tokens**. Alternativa monolítica = 60 × 500 = **30 000 tokens en cada
llamada**. La divulgación progresiva paga ~15 % y solo cuando se usa; además el
prompt monolítico degrada el seguimiento de instrucciones mucho antes de agotar la
ventana.

**Ejercicio 2.** Lo evaluable: description con *qué* + *disparadores* ("valida CSV
antes de analizar; úsalo cuando el usuario entregue un .csv o pida limpieza de
datos"), un paso determinista delegado a `scripts/check.py` y un contrato de salida
explícito (p. ej. `{filas, columnas, problemas: [...], apto: bool}`).

**Ejercicio 3.** (a) **script** — conteo determinista. (b) **script** con heurística
fija (tipos de la fila 1 vs. resto); documentar el criterio. (c) **instrucciones** —
redacción para humanos, requiere adaptar tono y contexto. (d) **script** — regex/parse
exacto. (e) **instrucciones** — depende del análisis pedido: es criterio, no regla.

**Ejercicio 4.** La traza muestra `status` → `sum` → final. Una description válida:
"Verifica el estado de un servicio y computa la suma solicitada, reportando la traza
completa de tool calls. Úsalo cuando pidan 'chequea el servicio y calcula X' o
verificaciones de salud con un cálculo asociado." — nombra las dos acciones y los
disparadores, sin describir la implementación.


In [ ]:
result = run_lab("agent", seed=130)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1
tokens_metadatos = 60 * 35
tokens_activados = 400 + 700 + 1200
total_skills = tokens_metadatos + tokens_activados
total_prompt_sistema = 60 * 500
print(f"skills: {total_skills} tokens | monolítico: {total_prompt_sistema} tokens "
      f"({total_skills / total_prompt_sistema:.0%})")

# Ejercicio 2 (referencia)
skill_md = '''---
name: csv-sanity-check
description: Valida un archivo CSV antes de cualquier análisis (estructura,
  cabecera, tipos, nulos). Úsalo cuando el usuario entregue un .csv o pida
  "limpiar datos" / "revisar el dataset".
---
# CSV sanity check
1. Ejecuta scripts/check.py <ruta>: devuelve {filas, columnas, cabecera,
   fechas_invalidas, pct_nulos}.
2. Con criterio: decide si el pct_nulos es aceptable PARA el análisis pedido.
3. Redacta el resumen de problemas en lenguaje claro.
4. Contrato de salida: {apto: bool, problemas: [...], evidencia: {...}}.
'''
print(skill_md.splitlines()[2])

# Ejercicio 4
result = run_lab("agent", seed=130)
assert result["kind"] == "agent"
herramientas = [paso["action"]["tool"] for paso in result["result"]["trace"]]
print("traza de tools:", herramientas, "→ final:", result["result"]["final"])


## Reflexión

1. El laboratorio (`run_lab("agent")`) muestra un agente con tools `status` y `sum`. Si empaquetaras su objetivo como skill, ¿qué irías a instrucciones, qué a script y qué quedaría como tool MCP?
2. ¿Por qué la description del skill importa más que el cuerpo para el comportamiento del sistema completo, si es la parte más corta?
3. Un skill de terceros incluye `scripts/limpiar.sh` que pide permisos de red. ¿Qué preguntas harías antes de instalarlo y qué principio de la clase 128 (contratos) estás aplicando?
